In [18]:
import pandas as pd
import numpy as np
from sklearn.svm import NuSVR
from sklearn.preprocessing import normalize
from scipy.optimize import nnls
from datetime import datetime
import warnings
warnings.filterwarnings('ignore')

print("All imports successful")

All imports successful


In [19]:
import os
os.chdir('/Users/parthshringarpure/Desktop/AI/Projects/luad_survival')

# Load full expression matrix (478 patients × 20,502 genes)
# We need the full matrix — not the filtered 1,000 — for CIBERSORT
expr_full = pd.read_csv('data/processed/expression_full_478.csv', index_col=0)

# Load LM22 signature matrix (547 genes × 22 cell types)
lm22 = pd.read_csv('data/external/LM22.txt', sep='\t', index_col=0)

print(f"Expression matrix: {expr_full.shape}  (patients × genes)")
print(f"LM22 matrix:       {lm22.shape}  (genes × cell types)")

# Find overlapping genes
common_genes = lm22.index.intersection(expr_full.columns)
print(f"\nOverlapping genes: {len(common_genes)} / {len(lm22.index)} ({len(common_genes)/len(lm22.index)*100:.1f}%)")

# Subset to common genes — aligned
expr_common = expr_full[common_genes]
lm22_common = lm22.loc[common_genes]

print(f"\nexpr_common shape: {expr_common.shape}")
print(f"lm22_common shape: {lm22_common.shape}")
print(f"Genes aligned: {list(expr_common.columns[:3]) == list(lm22_common.index[:3])}")

Expression matrix: (478, 20502)  (patients × genes)
LM22 matrix:       (547, 22)  (genes × cell types)

Overlapping genes: 513 / 547 (93.8%)

expr_common shape: (478, 513)
lm22_common shape: (513, 22)
Genes aligned: True


In [20]:
from sklearn.svm import NuSVR
from sklearn.preprocessing import normalize
from scipy.optimize import nnls

def run_cibersort_single(patient_expr, lm22_matrix):
    """
    DIY CIBERSORT — validated version.
    
    Fixes applied:
    1. Back-transform log2 to linear space
    2. Tune nu across [0.25, 0.5, 0.75]
    3. Non-negativity clip + NNLS fallback
    4. Sum-to-one normalisation
    Note: unit-norm on intersecting genes only (correct FIX 5)
    """
    
    # FIX 1: Back-transform log2 to linear space
    expr_linear = (2 ** patient_expr.values) - 1
    expr_linear = np.clip(expr_linear, 0, 1e6)
    
    lm22_linear = (2 ** lm22_matrix.values) - 1
    lm22_linear = np.clip(lm22_linear, 0, 1e6)
    
    # FIX 5 (correct): normalise ONLY the 513 intersecting genes
    # unit norm preserves relative expression ratios between cell types
    lm22_norm = normalize(lm22_linear, axis=0)   # each cell type column → unit norm
    expr_norm  = normalize(expr_linear.reshape(1, -1))[0]  # patient → unit norm
    
    # FIX 2: Tune nu parameter per patient
    best_nu    = 0.5
    best_error = np.inf
    
    for nu in [0.25, 0.5, 0.75]:
        try:
            svr = NuSVR(nu=nu, kernel='linear', C=1.0)
            svr.fit(lm22_norm, expr_norm)
            pred  = lm22_norm @ svr.coef_[0]
            error = np.mean((pred - expr_norm) ** 2)
            if error < best_error:
                best_error = error
                best_nu    = nu
        except:
            continue
    
    # Final fit with best nu
    try:
        svr = NuSVR(nu=best_nu, kernel='linear', C=1.0)
        svr.fit(lm22_norm, expr_norm)
        raw_weights = svr.coef_[0]
    except:
        raw_weights = np.zeros(lm22_matrix.shape[1])
    
    # FIX 3: Clip negatives, NNLS fallback if all zero
    clipped = np.maximum(raw_weights, 0)
    
    if clipped.sum() == 0:
        clipped, _ = nnls(lm22_norm, expr_norm)
        clipped = np.maximum(clipped, 0)
    
    # FIX 4: Sum-to-one normalisation
    total = clipped.sum()
    final = clipped / total if total > 0 else np.ones(len(clipped)) / len(clipped)
    
    return dict(zip(lm22_matrix.columns, final))

print("CIBERSORT function defined successfully")

CIBERSORT function defined successfully


In [21]:
test_patient = expr_common.iloc[0]
test_id      = expr_common.index[0]

print(f"Testing on patient: {test_id}")
print("Running improved CIBERSORT...")

fractions = run_cibersort_single(test_patient, lm22_common)

print(f"\nImmune fractions for {test_id}:")
for cell_type, frac in fractions.items():
    bar = '█' * int(frac * 40)
    print(f"  {cell_type:<35} {frac:.4f}  {bar}")

print(f"\nFractions sum to: {sum(fractions.values()):.6f}  (should be ~1.0)")

Testing on patient: tcga-05-4249
Running improved CIBERSORT...

Immune fractions for tcga-05-4249:
  B cells naive                       0.0000  
  B cells memory                      0.0322  █
  Plasma cells                        0.0171  
  T cells CD8                         0.0000  
  T cells CD4 naive                   0.0000  
  T cells CD4 memory resting          0.0396  █
  T cells CD4 memory activated        0.0000  
  T cells follicular helper           0.0917  ███
  T cells regulatory (Tregs)          0.0000  
  T cells gamma delta                 0.0000  
  NK cells resting                    0.0000  
  NK cells activated                  0.0287  █
  Monocytes                           0.1177  ████
  Macrophages M0                      0.0892  ███
  Macrophages M1                      0.0757  ███
  Macrophages M2                      0.2898  ███████████
  Dendritic cells resting             0.1033  ████
  Dendritic cells activated           0.0437  █
  Mast cells resting   

In [22]:
print(f"Starting improved CIBERSORT on {len(expr_common)} patients...")
print(f"Start time: {datetime.now().strftime('%H:%M:%S')}")
print("This will take longer than before due to nu tuning...\n")

results = {}

for i, patient_id in enumerate(expr_common.index):
    patient_expr = expr_common.loc[patient_id]
    results[patient_id] = run_cibersort_single(patient_expr, lm22_common)
    
    if (i + 1) % 50 == 0 or i == 0:
        print(f"  [{datetime.now().strftime('%H:%M:%S')}]  {i+1}/{len(expr_common)} patients done")

print(f"\nDone! End time: {datetime.now().strftime('%H:%M:%S')}")
print(f"Results computed for {len(results)} patients")

Starting improved CIBERSORT on 478 patients...
Start time: 15:40:34
This will take longer than before due to nu tuning...

  [15:40:34]  1/478 patients done
  [15:40:35]  50/478 patients done
  [15:40:36]  100/478 patients done
  [15:40:36]  150/478 patients done
  [15:40:37]  200/478 patients done
  [15:40:38]  250/478 patients done
  [15:40:38]  300/478 patients done
  [15:40:39]  350/478 patients done
  [15:40:39]  400/478 patients done
  [15:40:40]  450/478 patients done

Done! End time: 15:40:40
Results computed for 478 patients


In [23]:
immune_df = pd.DataFrame(results).T
immune_df.index.name = 'patient_id'

print(f"Shape: {immune_df.shape}")
print(f"\nAll fractions sum to 1.0?")
row_sums = immune_df.sum(axis=1)
print(f"  Min row sum:  {row_sums.min():.6f}")
print(f"  Max row sum:  {row_sums.max():.6f}")
print(f"  Mean row sum: {row_sums.mean():.6f}")

print(f"\nAny negative values? {(immune_df < 0).any().any()}")
print(f"Any NaN values?      {immune_df.isna().any().any()}")

print(f"\nMean immune composition across all 478 patients:")
mean_fracs = immune_df.mean().sort_values(ascending=False)
for cell_type, frac in mean_fracs.items():
    bar = '█' * int(frac * 60)
    print(f"  {cell_type:<35} {frac:.4f}  {bar}")

Shape: (478, 22)

All fractions sum to 1.0?
  Min row sum:  1.000000
  Max row sum:  1.000000
  Mean row sum: 1.000000

Any negative values? False
Any NaN values?      False

Mean immune composition across all 478 patients:
  Macrophages M2                      0.1448  ████████
  Dendritic cells resting             0.1194  ███████
  Mast cells resting                  0.1022  ██████
  Macrophages M1                      0.0786  ████
  Dendritic cells activated           0.0778  ████
  Monocytes                           0.0742  ████
  Macrophages M0                      0.0712  ████
  NK cells activated                  0.0583  ███
  T cells CD4 memory activated        0.0447  ██
  T cells follicular helper           0.0415  ██
  T cells CD8                         0.0358  ██
  T cells regulatory (Tregs)          0.0281  █
  T cells CD4 memory resting          0.0269  █
  Eosinophils                         0.0226  █
  T cells CD4 naive                   0.0195  █
  B cells naive      

In [24]:
# Overwrite old file with improved version
immune_df.to_csv('data/processed/immune_features_cibersort.csv')

print(f"Saved: data/processed/immune_features_cibersort.csv")
print(f"Shape: {immune_df.shape}")
print(f"\nThis replaces the original version with:")
print(f"  - Log2 back-transform to linear space")
print(f"  - Nu parameter tuning per patient [0.25, 0.5, 0.75]")
print(f"  - NNLS fallback for zero-weight patients")
print(f"  - Sum-to-one normalisation")

Saved: data/processed/immune_features_cibersort.csv
Shape: (478, 22)

This replaces the original version with:
  - Log2 back-transform to linear space
  - Nu parameter tuning per patient [0.25, 0.5, 0.75]
  - NNLS fallback for zero-weight patients
  - Sum-to-one normalisation
